In [15]:
from pathlib import Path

def get_mp4_stems(folder_path, recursive=True):
    folder = Path(folder_path)
    files = folder.rglob("*.mp4") if recursive else folder.glob("*.mp4")
    return [file.stem for file in files]

def get_json_stems(folder_path, recursive=True):
    folder = Path(folder_path)
    files = folder.rglob("*.json") if recursive else folder.glob("*.json")
    return [file.stem for file in files]

# 三个视频文件夹里的 mp4 名字（不带后缀）
folder1_list = get_mp4_stems("/mnt/mydisk/download_youtube_video/downloaded_video_surglavi")
folder2_list = get_mp4_stems("/mnt/mydisk/download_youtube_video/downloaded_video")
folder3_list = get_mp4_stems("/mnt/mydisk/download_youtube_video/downloaded_video_l")

mp4_names = set(folder1_list) | set(folder2_list) | set(folder3_list)

# json 文件名（不带后缀）
json_names = get_json_stems("/mnt/mydisk/Surglavi_videos/surglavi_narration_text")

# 检查每个 json 是否以前缀匹配某个 mp4
matched_json = []
unmatched_json = []

for json_name in json_names:
    if any(json_name.startswith(mp4_name) for mp4_name in mp4_names):
        matched_json.append(json_name)
    else:
        unmatched_json.append(json_name)

print("mp4总数:", len(mp4_names))
print("json总数:", len(json_names))
print("能在三个mp4文件夹中找到前缀匹配的json数:", len(matched_json))
print("找不到对应mp4前缀的json数:", len(unmatched_json))

print("\n前10个找不到对应mp4的json:")
print(unmatched_json[:10])

mp4总数: 22313
json总数: 3148
能在三个mp4文件夹中找到前缀匹配的json数: 2606
找不到对应mp4前缀的json数: 542

前10个找不到对应mp4的json:
['QJnJ8JVplaM', 'wKYIoT9_Aig', 'H1nrOyJjTeM', 'tFo1R4do0gA', 'UKmnYLjZmGU', 'uL6lYILBeOE', 'YH5CWLZ1DmU', 'zIWNA03WxrM', 'a9QjxojPyfs', 'iBzZsuXui5Q']


In [1]:
# -*- coding: utf-8 -*-
import os
import json
import glob
import numpy as np

# ============================================================
# 你只需要改下面 3 个路径/配置
# ============================================================

# 1) 你的每视频一个 json 的目录（里面是 VID06.json, VID51.json ...）
JSON_DIR = "/mnt/mydisk/cholecdata/cholecdata/cholect50/labels"

# 2) 官方的 label_mapping.txt（6列：triplet,instrument,verb,target,iv,it）
LABEL_MAPPING_PATH = "/mnt/mydisk/cholecdata/cholecdata/cholect50/label_mapping.txt"

# 3) 输出目录（会创建 instrument/ verb/ target/ triplet/ 四个子目录）
OUT_DIR = "/mnt/mydisk/CLIP/CholecT50/labels"

# 额外：json 文件匹配模式
JSON_GLOB = "VID*.json"

# 严格模式：遇到越界/缺映射直接报错（建议 True，保证生成结果靠谱）
STRICT = True

# 你这种 instance list 的 triplet_id 位置（根据你贴的示例，默认第0位是 triplet_id）
TRIPLET_ID_INDEX_IN_INSTANCE = 0

# ============================================================


def load_json(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def infer_num_classes(meta: dict):
    """
    从 categories 推断类别数（保证维度和官方类别表一致）
    """
    cats = meta.get("categories", {})

    def n(key):
        d = cats.get(key, {})
        return len(d) if isinstance(d, dict) else len(list(d))

    return n("instrument"), n("verb"), n("target"), n("triplet")


def load_label_mapping(mapping_path: str):
    """
    label_mapping.txt: 6列
      col0 triplet_id
      col1 instrument_id
      col2 verb_id
      col3 target_id
      col4 iv_id
      col5 it_id
    """
    m = np.loadtxt(mapping_path, dtype=np.int32, delimiter=",")
    if m.ndim == 1:
        m = m[None, :]
    if m.shape[1] < 4:
        raise ValueError(f"label_mapping must have >=4 columns, got {m.shape[1]}")

    t2i, t2v, t2t = {}, {}, {}
    for row in m:
        tid = int(row[0])
        t2i[tid] = int(row[1])
        t2v[tid] = int(row[2])
        t2t[tid] = int(row[3])
    return t2i, t2v, t2t


def get_triplet_id_from_instance(inst):
    if inst is None or len(inst) == 0:
        return -1
    idx = TRIPLET_ID_INDEX_IN_INSTANCE
    if idx < 0 or idx >= len(inst):
        return -1
    return int(inst[idx])


def save_txt(mat: np.ndarray, out_path: str):
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    np.savetxt(out_path, mat, fmt="%d", delimiter=",")


def build_and_save_for_one_video(json_path: str, t2i, t2v, t2t):
    meta = load_json(json_path)
    vid_name = os.path.splitext(os.path.basename(json_path))[0]  # e.g. VID06

    anns = meta.get("annotations", {})
    if not isinstance(anns, dict) or len(anns) == 0:
        raise ValueError(f"{vid_name}: annotations missing/empty in {json_path}")

    num_i, num_v, num_t, num_ivt = infer_num_classes(meta)

    frame_ids = sorted(int(k) for k in anns.keys())
    N = len(frame_ids)

    tool_mat = np.zeros((N, 1 + num_i), dtype=np.int32)
    verb_mat = np.zeros((N, 1 + num_v), dtype=np.int32)
    target_mat = np.zeros((N, 1 + num_t), dtype=np.int32)
    triplet_mat = np.zeros((N, 1 + num_ivt), dtype=np.int32)

    missing_map = 0

    for r, fid in enumerate(frame_ids):
        tool_mat[r, 0] = fid
        verb_mat[r, 0] = fid
        target_mat[r, 0] = fid
        triplet_mat[r, 0] = fid

        inst_list = anns[str(fid)]
        for inst in inst_list:
            tid = get_triplet_id_from_instance(inst)
            if tid == -1:
                continue

            if not (0 <= tid < num_ivt):
                if STRICT:
                    raise ValueError(f"{vid_name}: triplet_id out of range tid={tid} at frame={fid}, expect [0,{num_ivt-1}]")
                continue

            triplet_mat[r, 1 + tid] = 1

            if tid not in t2i:
                missing_map += 1
                if STRICT:
                    raise KeyError(f"{vid_name}: triplet_id {tid} not in label_mapping (frame={fid})")
                continue

            iid = t2i[tid]
            vid = t2v[tid]
            taid = t2t[tid]

            if 0 <= iid < num_i:
                tool_mat[r, 1 + iid] = 1
            elif STRICT:
                raise ValueError(f"{vid_name}: instrument_id {iid} out of range for triplet {tid}")

            if 0 <= vid < num_v:
                verb_mat[r, 1 + vid] = 1
            elif STRICT:
                raise ValueError(f"{vid_name}: verb_id {vid} out of range for triplet {tid}")

            if 0 <= taid < num_t:
                target_mat[r, 1 + taid] = 1
            elif STRICT:
                raise ValueError(f"{vid_name}: target_id {taid} out of range for triplet {tid}")

    save_txt(tool_mat, os.path.join(OUT_DIR, "instrument", f"{vid_name}.txt"))
    save_txt(verb_mat, os.path.join(OUT_DIR, "verb", f"{vid_name}.txt"))
    save_txt(target_mat, os.path.join(OUT_DIR, "target", f"{vid_name}.txt"))
    save_txt(triplet_mat, os.path.join(OUT_DIR, "triplet", f"{vid_name}.txt"))

    print(f"[{vid_name}] OK  frames={N}  classes(i/v/t/ivt)={num_i}/{num_v}/{num_t}/{num_ivt}  missing_map={missing_map}")
    print(f"  -> {OUT_DIR}/instrument/{vid_name}.txt")
    print(f"  -> {OUT_DIR}/verb/{vid_name}.txt")
    print(f"  -> {OUT_DIR}/target/{vid_name}.txt")
    print(f"  -> {OUT_DIR}/triplet/{vid_name}.txt")


def main():
    os.makedirs(OUT_DIR, exist_ok=True)

    if not os.path.exists(JSON_DIR):
        raise FileNotFoundError(f"JSON_DIR not found: {JSON_DIR}")
    if not os.path.exists(LABEL_MAPPING_PATH):
        raise FileNotFoundError(f"LABEL_MAPPING_PATH not found: {LABEL_MAPPING_PATH}")

    t2i, t2v, t2t = load_label_mapping(LABEL_MAPPING_PATH)

    json_paths = sorted(glob.glob(os.path.join(JSON_DIR, JSON_GLOB)))
    if not json_paths:
        raise FileNotFoundError(f"No json files found under {JSON_DIR} with glob {JSON_GLOB}")

    print(f"Found {len(json_paths)} videos under {JSON_DIR}")
    for jp in json_paths:
        build_and_save_for_one_video(jp, t2i, t2v, t2t)

    print("\nDone.")


if __name__ == "__main__":
    main()

Found 50 videos under /mnt/mydisk/cholecdata/cholecdata/cholect50/labels
[VID01] OK  frames=1734  classes(i/v/t/ivt)=6/10/15/100  missing_map=0
  -> /mnt/mydisk/CLIP/CholecT50/labels/instrument/VID01.txt
  -> /mnt/mydisk/CLIP/CholecT50/labels/verb/VID01.txt
  -> /mnt/mydisk/CLIP/CholecT50/labels/target/VID01.txt
  -> /mnt/mydisk/CLIP/CholecT50/labels/triplet/VID01.txt
[VID02] OK  frames=2840  classes(i/v/t/ivt)=6/10/15/100  missing_map=0
  -> /mnt/mydisk/CLIP/CholecT50/labels/instrument/VID02.txt
  -> /mnt/mydisk/CLIP/CholecT50/labels/verb/VID02.txt
  -> /mnt/mydisk/CLIP/CholecT50/labels/target/VID02.txt
  -> /mnt/mydisk/CLIP/CholecT50/labels/triplet/VID02.txt
[VID04] OK  frames=1523  classes(i/v/t/ivt)=6/10/15/100  missing_map=0
  -> /mnt/mydisk/CLIP/CholecT50/labels/instrument/VID04.txt
  -> /mnt/mydisk/CLIP/CholecT50/labels/verb/VID04.txt
  -> /mnt/mydisk/CLIP/CholecT50/labels/target/VID04.txt
  -> /mnt/mydisk/CLIP/CholecT50/labels/triplet/VID04.txt
[VID05] OK  frames=2345  classes(